# 02 — A planet around a mass-losing star

**Physics:** slow isotropic mass loss only. We use the saved pilot from notebook 01 and the analytical response

$$a(t)=a_0\frac{M_{\star,0}+m_p}{M_\star(t)+m_p}.$$

We compare the pericentre with the stellar radius to look for geometric crossing. Tides and the brown dwarf are still off; this does not settle engulfment or survival.

This notebook should take seconds: it reads a saved track and starts no AMUSE workers. The saved plots below use the original pilot; later refinements are checked in notebooks 01 and 04.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from amuse.units import units


# Keep the pilot for this simple example. Change this relative path to try another saved track.
STELLAR_TRACK = "data/stellar_track_pilot.npz"
PROJECT_ROOT = None  # Normally found from the notebook's working directory.


def locate_project_root(project_root=None):
    if project_root is not None:
        candidate = Path(project_root).expanduser().resolve()
        if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
            return candidate
        raise FileNotFoundError(f"Not a white_dwarf_project folder: {candidate}")

    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "white_dwarf_project"):
            if (candidate / "notebooks" / "01_stellar_evolution.ipynb").is_file():
                return candidate
    raise FileNotFoundError(
        "Open this notebook from the cloned project, "
        "or set PROJECT_ROOT to your white_dwarf_project folder."
    )


ROOT = locate_project_root(PROJECT_ROOT)
relative = Path(STELLAR_TRACK)
track_path = (ROOT / relative).resolve()
if relative.is_absolute() or not track_path.is_relative_to(ROOT):
    raise ValueError("Use a stellar-track path relative to white_dwarf_project.")
if not track_path.is_file():
    raise FileNotFoundError(
        f"Missing stellar track: {STELLAR_TRACK}. "
        "Run notebook 01's pilot section, or copy the saved track into this project."
    )
FIGURE_DIR = ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

with np.load(track_path, allow_pickle=False) as loaded:
    track = {key: loaded[key].copy() for key in loaded.files}

age_Gyr = track["time_Myr"] / 1000
mass_MSun = track["mass_MSun"]
radius_AU = track["radius_RSun"] * (1 | units.RSun).value_in(units.au)
print("Using:", track_path.relative_to(ROOT).as_posix())
print("Loaded SeBa stellar track with", len(age_Gyr), "outputs")

## 1. Mass-loss-driven orbital expansion

In [ ]:
planet_mass_MSun = (1.9 | units.MEarth).value_in(units.MSun)
# Try a few starting distances around the giant-star scale.
initial_semimajor_axes_AU = np.array([0.6, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0])
initial_eccentricity = 0.0  # Circular for this first comparison.

# Adiabatic response. Eccentricity is constant in this approximation.
expansion_factor = (mass_MSun[0] + planet_mass_MSun) / (mass_MSun + planet_mass_MSun)

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
ax.plot(age_Gyr, radius_AU, color="black", linewidth=2, label="Stellar radius")

for a0 in initial_semimajor_axes_AU:
    a = a0 * expansion_factor
    q = a * (1 - initial_eccentricity)
    indices = np.flatnonzero(q <= radius_AU)
    label = f"$a_0={a0:g}$ au"
    line, = ax.plot(age_Gyr, q, label=label)
    if len(indices):
        i = int(indices[0])
        ax.scatter(age_Gyr[i], q[i], color=line.get_color(), marker="x", s=45)

ax.set_yscale("log")
ax.set_xlabel("Time since ZAMS [Gyr]")
ax.set_ylabel("Distance [au]")
ax.set_title("Adiabatic orbital expansion versus stellar envelope")
ax.legend(ncol=2, frameon=False, fontsize=9)
ax.grid(alpha=0.2)

# Focusing on late stellar evolution while still keeping the main sequence visible.
giant = np.flatnonzero(np.isin(track["stellar_type"], [2, 3, 4, 5, 6]))
if len(giant):
    ax.set_xlim(max(0, age_Gyr[giant[0]] - 0.1), age_Gyr[-1] + 0.05)

figure_path = FIGURE_DIR / "isolated_planet_pilot.pdf"
fig.savefig(figure_path, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)

## 2. Geometric crossing boundary

For fixed eccentricity, crossing occurs when

$$a_0(1-e_0)\frac{M_{\star,0}+m_p}{M_\star(t)+m_p}\le R_\star(t).$$

Solve for $a_0$ at each saved time, then take the largest value. An initial orbit above that boundary avoids radius crossing at all saved points in this model.

In [ ]:
# Calculate this once for the boundary, its peak and the plot below.
critical_distance = (
    radius_AU * (mass_MSun + planet_mass_MSun)
    / ((mass_MSun[0] + planet_mass_MSun) * (1 - initial_eccentricity))
)
i_critical = int(np.argmax(critical_distance))
a_critical_AU = float(critical_distance[i_critical])
print(f"Pilot geometric crossing boundary: a0 > {a_critical_AU:.3f} au")
print("This excludes tides, brown-dwarf perturbations, and detailed envelope physics.")

print("\nInitial orbit outcomes (sampled stellar track):")
for a0 in initial_semimajor_axes_AU:
    q = a0 * (1 - initial_eccentricity) * expansion_factor
    crossing = np.flatnonzero(q <= radius_AU)
    label = f"crosses envelope at {age_Gyr[crossing[0]]:.3f} Gyr" if len(crossing) else "no sampled envelope crossing"
    print(f"  a0 = {a0:4.1f} au: {label}")

# Inspect the most restrictive saved state.
print("Most restrictive point in stellar evolution:")
print(f"Time: {age_Gyr[i_critical]:.6f} Gyr")
print(f"Stellar mass: {mass_MSun[i_critical]:.4f} Msun")
print(f"Stellar radius: {radius_AU[i_critical]:.4f} AU")
print(f"Orbital expansion factor: {expansion_factor[i_critical]:.4f}")
print(f"Critical initial distance: {critical_distance[i_critical]:.4f} AU")
print(f"Stellar type: {track['stellar_type'][i_critical]}")

In [ ]:
# The maximum of this curve sets the boundary for the saved track.
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    age_Gyr,
    critical_distance,
    color="black",
    linewidth=2,
    label="Instantaneous critical initial distance"
)

ax.axhline(
    a_critical_AU,
    color="red",
    linestyle="--",
    label=f"Overall boundary = {a_critical_AU:.3f} AU"
)

ax.set_xlabel("Time since ZAMS [Gyr]")
ax.set_ylabel("Critical initial semimajor axis [AU]")

ax.set_xlim(5.5, age_Gyr[-1] + 0.02)

ax.legend(frameon=False)

plt.show()